# Price elasticity and real battery revenue

**Summary.** The benchmark treats its 1 MW battery as a price-taker, so it collects the full spot price even in scarcity intervals where a few hundred MW of extra supply would collapse the price. Using AEMO's own P5MIN price sensitivities, a fleet of only 100–200 MW running the TWAP baseline's dispatch would earn about 35% less on the mainland (59% less in thin South Australia), which closes most of the gap between our baseline and what the real NEM battery fleet earned. For this project, comparisons between policies stay like-for-like (every policy and the perfect-foresight reference face the same prices), but the benchmark's \\$/MW/yr figures are an upper bound for a marginal battery rather than achievable fleet revenue, and the score rewards spike capture that a real fleet could not all collect at once.

## What this notebook does

The question started from the leaderboard: the TWAP baseline's \\$/MW/yr looked high next to the fleet revenue on [nembess.com/trends](https://nembess.com/trends). This notebook compares like with like and then measures how much of the difference is price impact.

**Comparison basis.** Real revenue only: gross trading revenue, $\sum_t (\text{discharged}_t - \text{charged}_t) \times \text{price}_t$, with **no wear cost and no stored-energy (terminal) value**, which are part of the benchmark's objective but not of anyone's revenue. Prices are **unmapped**: the benchmark stretches historical prices above \\$1,000 so each period's market price cap lands on today's \\$23,200, which would flatter us against real batteries that were paid the price of the day.

**Data.**
- The benchmark's TWAP reference policy, simulated over the 12 v2 holdout months (one of each calendar month, Jan 2023 – Jun 2026) in all five regions, once on mapped and once on unmapped prices. Battery: 1 MW / 2 MWh, 90% one-way efficiency, 5–95% state of charge (1.8 MWh usable).
- nembess.com's fleet revenue per MW (its API returns \\$/MW/day; the site multiplies by 365), split into energy, regulation FCAS and contingency FCAS. Tasmania has no grid batteries there, so fleet comparisons are mainland only.
- AEMO's P5MIN price sensitivities and their scenario definitions (`P5MIN_SCENARIODEMAND`), for price impact.

Everything is cached in `data/analysis/price_elasticity/` (gitignored); a cell recomputes whatever is missing. The two TWAP simulations take about 3 minutes each.

In [ ]:
from pathlib import Path
import io, json, sys, zipfile

import duckdb
import httpx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import harness.build_data as build_data
from bess.config import load_config
from bess.normalise import PriceMap
from harness.evaluator import REFERENCE_POLICY_DIR, evaluate
from harness.splits import HOLDOUT_MONTHS

CACHE = ROOT / "data" / "analysis" / "price_elasticity"
CACHE.mkdir(parents=True, exist_ok=True)
RAW_DB = ROOT / "data" / "benchmark.duckdb"       # AEMO tables as published
MAPPED_DB = build_data.DB_PATH                     # what the benchmark scores on
UNMAPPED_DB = CACHE / "market_unmapped.duckdb"     # same tables, prices left at their historical values
REGIONS = ["NSW1", "QLD1", "SA1", "VIC1", "TAS1"]
MAINLAND = ["NSW1", "QLD1", "SA1", "VIC1"]
ENERGY_MWH, USABLE_MWH = 2.0, 1.8
TO_K_PER_YEAR = 365 / 1000                         # $/MW/day -> $k/MW/yr

# chart style: the dataviz reference palette, light mode
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e0"
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
REGION_COLOUR = {"NSW1": BLUE, "QLD1": ORANGE, "SA1": AQUA, "VIC1": YELLOW, "TAS1": MAGENTA}
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "lines.linewidth": 2, "lines.markersize": 7, "legend.frameon": False, "font.size": 10.5,
    "figure.dpi": 110,
})
pd.set_option("display.precision", 1)

## 1. Data

The unmapped database is built from the raw AEMO tables with the benchmark's own builder, using a cap mapping whose knee no price reaches, so every price stays as published. TWAP then runs over the holdout months on both databases, keeping each interval's dispatch.

In [ ]:
if not UNMAPPED_DB.exists():
    build_data.PRICE_MAP = PriceMap(mpc_now=build_data.PRICE_MAP.mpc_now, knee=1e12)
    build_data.build(dest=UNMAPPED_DB, raw=RAW_DB)


def twap_intervals(name: str, db: Path) -> pd.DataFrame:
    out = CACHE / f"twap_{name}"
    if not (out / "done.json").exists():
        res = evaluate(REFERENCE_POLICY_DIR, load_config(), HOLDOUT_MONTHS, db_path=db,
                       sandboxed=False, timeout_s=7200, out_dir=out)
        assert all(j.get("valid") for j in res["jobs"])
        (out / "done.json").write_text(json.dumps({"pct_of_perfect": res["pct_of_perfect"]}))
    frames = [pd.read_parquet(f, columns=["interval_end", "discharged_mwh", "charged_mwh", "rrp", "revenue"])
              .assign(region=f.name.split("_")[0]) for f in sorted(out.glob("*_intervals.parquet"))]
    d = pd.concat(frames, ignore_index=True)
    start = d.interval_end - pd.Timedelta(minutes=5)          # interval_end -> the interval's own day and month
    return d.assign(day=start.dt.date, month=start.dt.strftime("%Y-%m"))


twap = twap_intervals("unmapped", UNMAPPED_DB)
twap_mapped = twap_intervals("mapped", MAPPED_DB)
MONTHS = sorted(twap.month.unique())
print(f"{len(twap):,} intervals, {twap.region.nunique()} regions, months: {', '.join(MONTHS)}")

In [ ]:
NEMBESS = "https://api.nembess.com/api"


def cached_json(name: str, url: str, **params):
    f = CACHE / name
    if not f.exists():
        r = httpx.get(url, params=params, timeout=60)
        r.raise_for_status()
        f.write_text(r.text)
    return json.loads(f.read_text())


rows = []
for region in MAINLAND:
    js = cached_json(f"nembess_{region}.json", f"{NEMBESS}/trends/revenue",
                     group="service", unit="per_mw", region=region, window="ALL")
    for p in js["points"]:
        if p["series"]["energy"] is not None:
            rows.append({"region": region, "month": p["t"][:7],
                         **{k: (p["series"][k] or 0) * TO_K_PER_YEAR for k in ("energy", "regulation", "contingency", "total")}})
fleet = pd.DataFrame(rows).set_index(["region", "month"])

batteries = pd.DataFrame(cached_json("nembess_batteries.json", f"{NEMBESS}/batteries"))
fleet_size = (batteries[batteries.current_power_capacity > 0]
              .groupby("region_id")[["current_power_capacity", "current_storage_capacity"]].sum()
              .rename(columns={"current_power_capacity": "MW", "current_storage_capacity": "MWh"}))
fleet_size["hours"] = fleet_size.MWh / fleet_size.MW
fleet_size.round(1)

Today's fleet is 1.8–2.7 hours on average, about the same as our 2-hour battery, so shorter duration doesn't explain lower fleet revenue now (the 2023–24 fleet was smaller and shorter). Note that nembess's per-MW figures divide by installed capacity, including batteries still commissioning: NSW and QLD capacity roughly doubled from mid-2025, which depresses their recent per-MW revenue.

## 2. The benchmark's TWAP baseline against the real fleet

For each region and month: TWAP's gross revenue on mapped and unmapped prices, the **two-hour spread** heuristic, and the fleet's energy and energy-plus-FCAS revenue, all in \\$k/MW/yr.

The two-hour spread is, for each day, the average of the day's highest two hours of 5-minute prices (from any time of day) minus the average of its lowest two hours, times the battery's 2 MWh. It is the most a battery could make in a day from one cycle if it ignored losses and the need to charge before discharging. A useful yardstick, though not a strict cap: a battery that starts the day full, or cycles twice, can beat it.

In [ ]:
def per_year(d: pd.DataFrame, value: str) -> pd.Series:
    g = d.groupby(["region", "month"])
    return g[value].sum() / (g.size() / 288) * TO_K_PER_YEAR      # $ over the month -> $k/MW/yr


def two_hour_spread(p: pd.Series) -> float:
    s = np.sort(p.to_numpy())
    return s[-24:].mean() - s[:24].mean()                          # 24 five-minute intervals = 2 hours


spread = (twap.groupby(["region", "day", "month"]).rrp.apply(two_hour_spread)
          .groupby(["region", "month"]).mean() * ENERGY_MWH * TO_K_PER_YEAR)
table = pd.DataFrame({
    "TWAP, mapped prices": per_year(twap_mapped, "revenue"),
    "TWAP, actual prices": per_year(twap, "revenue"),
    "two-hour spread": spread,
}).join(fleet[["energy", "total"]].rename(columns={"energy": "fleet energy", "total": "fleet energy + FCAS"}))
monthly = table.loc[MAINLAND].groupby(level="month").mean()
monthly.loc["mean"] = monthly.mean()
monthly.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(MONTHS))
series = [
    ("two-hour spread", YELLOW, "-", "o"),
    ("TWAP, mapped prices", BLUE, "--", "o"),
    ("TWAP, actual prices", BLUE, "-", "o"),
    ("fleet energy + FCAS", AQUA, "-", "s"),
    ("fleet energy", ORANGE, "-", "s"),
]
for name, colour, ls, marker in series:
    y = monthly.loc[MONTHS, name].to_numpy()
    ax.plot(x, y, ls=ls, marker=marker, color=colour, label=name, markeredgecolor=SURFACE, markeredgewidth=1.5)
ax.set_yscale("log")
ax.set_yticks([10, 20, 50, 100, 200, 500])
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}k"))
ax.set_xticks(x, [pd.Timestamp(m).strftime("%b %y") for m in MONTHS])
ax.set_ylabel("$k per MW per year (log scale)")
ax.set_title("TWAP earns about twice the fleet's energy revenue, mostly in volatile months")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncols=5, fontsize=9)
plt.show()

In [ ]:
m = monthly.loc["mean"]
print(f"Mainland mean over the 12 holdout months ($k/MW/yr): TWAP {m['TWAP, actual prices']:.0f} on actual prices "
      f"({m['TWAP, mapped prices']:.0f} mapped), fleet energy {m['fleet energy']:.0f}, fleet energy + FCAS {m['fleet energy + FCAS']:.0f}, "
      f"two-hour spread {m['two-hour spread']:.0f}.")
print(f"TWAP / fleet energy: {m['TWAP, actual prices'] / m['fleet energy']:.1f}x;  TWAP / fleet energy + FCAS: "
      f"{m['TWAP, actual prices'] / m['fleet energy + FCAS']:.1f}x")
for name in ("TWAP, actual prices", "fleet energy", "fleet energy + FCAS"):
    print(f"share of the two-hour spread captured by {name}: {m[name] / m['two-hour spread']:.0%}")
by_region = table.groupby(level="region").mean()
by_region.loc[REGIONS].round(0)

**Reading it.**
- **Cap mapping is a minor part.** Moving to actual prices cuts TWAP's revenue by about 14% on the mainland; it matters most in 2023–24, when the cap was furthest below today's.
- **The gap is in volatile months.** In the quiet 2026 months TWAP and the fleet earn about the same. The 2.2× average comes from months with price spikes, above all August 2024.
- **FCAS explains 2023, not 2025.** In early 2023 most fleet revenue came from contingency FCAS: operators left energy revenue on the table for it. From mid-2024 FCAS revenue largely disappears, yet the fleet's energy revenue is still about half of TWAP's in Jul–Nov 2025.
- **Capture rates.** TWAP collects about half of the two-hour spread; the fleet about a quarter from energy, a bit over a third with FCAS.

## 3. How hard does TWAP cycle?

A cycle here is 1.8 MWh discharged, the battery's usable energy. TWAP bids around the forecast's time-weighted average price, so on flat days it barely trades.

In [ ]:
cycles = twap.groupby(["region", "day"]).discharged_mwh.sum() / USABLE_MWH
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(cycles, bins=np.arange(0, 2.05, 0.1), color=BLUE, edgecolor=SURFACE, linewidth=2)
ax.axvline(cycles.mean(), color=INK, lw=1.2, ls=":")
ax.annotate(f"mean {cycles.mean():.2f} cycles/day", (cycles.mean(), ax.get_ylim()[1] * 0.92), xytext=(6, 0),
            textcoords="offset points", color=INK2)
ax.set_xlabel("cycles per day")
ax.set_ylabel("region-days")
ax.set_title("TWAP averages about half a cycle a day")
plt.show()
stats = cycles.groupby(level="region").agg(mean="mean", min="min", max="max", no_discharge=lambda s: (s == 0).mean() * 100,
                                           over_one=lambda s: (s > 1).mean() * 100)
stats.loc["all"] = [cycles.mean(), cycles.min(), cycles.max(), (cycles == 0).mean() * 100, (cycles > 1).mean() * 100]
stats.rename(columns={"no_discharge": "% days idle", "over_one": "% days > 1 cycle"}).style.format(precision=2)

## 4. Where TWAP's revenue comes from

A large share of TWAP's revenue arrives in a handful of intervals above \\$1,000/MWh. August 2024 in NSW is mostly a single day: on 5 August 2024 NSW and VIC prices sat near the \\$17,500 cap twice, from 07:40 to 09:15 and from 17:15 to 19:10. TWAP started the day full, discharged into the morning spike, recharged at midday and discharged again in the evening.

In [ ]:
spike_share = (twap[twap.rrp > 1000].groupby("region").revenue.sum() / twap.groupby("region").revenue.sum() * 100)
day_rev = twap.groupby(["region", "day"]).revenue.sum()
top_day_share = day_rev.groupby(level="region").max() / twap.groupby("region").revenue.sum() * 100
pd.DataFrame({"% of revenue from intervals above $1,000": spike_share,
              "% of revenue from the single best day": top_day_share}).loc[REGIONS].round(0)

## 5. Measuring price impact with AEMO's price sensitivities

Every 5 minutes, AEMO's P5MIN pre-dispatch reruns each region with its demand shifted by fixed offsets and publishes the resulting prices (`P5MIN_PRICESENSITIVITIES`). The offsets are defined in `P5MIN_SCENARIODEMAND`. Lower demand by X MW is the same, for the price, as X MW of extra supply, such as a battery fleet discharging; higher demand is the same as X MW of extra charging. So each interval comes with an AEMO-solved answer to "what would the price have been with X MW more or less supply?".

We use each run's first interval (run time = interval time), which is the closest to real dispatch: its base price tracks the actual dispatch price closely. Sensitivity data starts in August 2024, so this covers 8 of the 12 holdout months.

In [ ]:
lines = (CACHE / "p5min_scenariodemand.csv")
if not lines.exists():
    url = ("https://www.nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM/2025/MMSDM_2025_10/"
           "MMSDM_Historical_Data_SQLLoader/DATA/PUBLIC_ARCHIVE%23P5MIN_SCENARIODEMAND%23FILE01%23202510010000.zip")
    z = zipfile.ZipFile(io.BytesIO(httpx.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60).content))
    lines.write_bytes(z.read(z.namelist()[0]))
raw = [l.split(",") for l in lines.read_text().splitlines()]
scenarios = pd.DataFrame([r[-5:] for r in raw if r[0] in ("D",) or r[0].startswith('"')],
                         columns=["effective", "version", "scenario", "region", "delta_mw"])
scenarios = scenarios.assign(scenario=scenarios.scenario.astype(int), delta_mw=scenarios.delta_mw.astype(float))
scenarios.pivot_table(index="region", columns="scenario", values="delta_mw").loc[REGIONS]

Each region has three scenarios: +X MW of demand, −X MW, and a larger positive one. We use the ±X pair: NSW ±200 MW, VIC and QLD ±150, SA ±100, TAS ±50 (scenario *n* is column `RRPn` in the sensitivities table).

In [ ]:
pairs = scenarios.groupby("region").apply(lambda g: pd.Series({
    "up": int(g.loc[g.delta_mw > 0].sort_values("delta_mw").scenario.iloc[0]),   # +X: extra charging
    "down": int(g.loc[g.delta_mw < 0].scenario.iloc[0]),                        # -X: extra supply
    "x_mw": float(-g.delta_mw[g.delta_mw < 0].iloc[0])}), include_groups=False)

con = duckdb.connect(str(RAW_DB), read_only=True)
sens = pd.concat([con.execute(f'''
    SELECT s.INTERVAL_DATETIME AS interval_end, s.REGIONID AS region, p.RRP AS base,
           s.RRP{int(r.up)} AS up, s.RRP{int(r.down)} AS down, {r.x_mw} AS x_mw
    FROM p5min_pricesensitivities s
    JOIN p5min_regionsolution p USING (RUN_DATETIME, INTERVAL_DATETIME, REGIONID, INTERVENTION)
    WHERE s.REGIONID = '{region}' AND s.INTERVENTION = 0 AND s.RUN_DATETIME = s.INTERVAL_DATETIME
      AND s.INTERVAL_DATETIME > '2024-08-01'
''').df() for region, r in pairs.iterrows()])
con.close()

EL_MONTHS = sorted(twap.month[twap.interval_end > sens.interval_end.min()].unique())
el = twap[twap.month.isin(EL_MONTHS)].merge(sens, on=["interval_end", "region"], how="left")
gaps = el.base.isna() | el.up.isna() | el.down.isna()
print(f"{len(el):,} intervals in {', '.join(EL_MONTHS)}; {gaps.sum()} have gaps in AEMO's P5MIN data "
      "and are treated as no price move")
el.loc[gaps, ["up", "down"]] = np.nan
el["base"] = el.base.fillna(el.rrp)
el["up"], el["down"] = el.up.fillna(el.base), el.down.fillna(el.base)
print(f"P5MIN base price vs actual dispatch price: median absolute difference ${np.median(np.abs(el.base - el.rrp)):.2f}/MWh")

### The 5 August 2024 evening in NSW

The actual NSW price, and the price P5MIN computed for the same interval with 200 MW more supply. Small amounts of extra supply would have taken the edge off the spike in many intervals; in others the price stays near the cap.

In [ ]:
day = el[(el.region == "NSW1") & (el.interval_end > "2024-08-05 06:00") & (el.interval_end <= "2024-08-05 20:00")]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
a1.step(day.interval_end, day.rrp, where="pre", color=BLUE, label="actual price")
a1.step(day.interval_end, day.down, where="pre", color=ORANGE, label="with 200 MW more supply (P5MIN)")
a1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
a1.set_ylabel("$/MWh")
a1.set_title("NSW, 5 Aug 2024: 200 MW more supply would often have collapsed the spike")
a1.legend(loc="center", ncols=1)
mw = (day.discharged_mwh - day.charged_mwh) * 12
a2.bar(day.interval_end, mw, width=pd.Timedelta(minutes=4), color=np.where(mw >= 0, BLUE, INK2))
a2.axhline(0, color=INK2, lw=0.8)
a2.set_ylabel("TWAP MW\n(+ discharge)")
a2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%H:%M"))
plt.show()

### How far does the price move?

For the intervals where TWAP discharged, the price drop per 100 MW of extra supply, grouped by the interval's price. At ordinary prices the supply stack is flat and 100 MW moves the price by a few dollars; in scarcity it moves by thousands. That asymmetry is why a lone price-taking battery does so well in spike months.

In [ ]:
dis = el[(el.discharged_mwh > 0) & (el.base > 0)].copy()
dis["drop_per_100mw"] = (dis.base - dis.down) / dis.x_mw * 100
dis["price band"] = pd.cut(dis.base, [0, 100, 300, 1000, 5000, np.inf], labels=["< $100", "$100–300", "$300–1k", "$1k–5k", "> $5k"])
bands = dis.groupby("price band", observed=True).drop_per_100mw.agg(intervals="size", median="median",
                                                                    p25=lambda s: s.quantile(.25), p75=lambda s: s.quantile(.75))
fig, ax = plt.subplots(figsize=(8, 4))
xb = np.arange(len(bands))
ax.bar(xb, bands["median"].clip(lower=0.5), width=0.6, color=BLUE)
ax.errorbar(xb, bands["median"].clip(lower=0.5), yerr=[(bands["median"] - bands.p25).clip(lower=0), bands.p75 - bands["median"]],
            fmt="none", ecolor=INK2, capsize=4, lw=1.2)
for i, (v, top) in enumerate(zip(bands["median"], bands.p75)):
    ax.annotate(f"${v:,.0f}", (i, top), xytext=(0, 6), textcoords="offset points", ha="center", color=INK2)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.set_xticks(xb, bands.index)
ax.set_xlabel("price in the interval")
ax.set_ylabel("price drop per 100 MW ($/MWh, log)")
ax.set_title("Scarcity prices are knife-edge (median, bars show 25th–75th percentile)")
plt.show()
bands.round(0)

## 6. Revenue if a fleet ran TWAP's dispatch

Suppose a fleet of F MW in a region did exactly what our 1 MW TWAP battery did. Discharging moves each interval's price down by F/X of the −X scenario's change, charging moves it up by F/X of the +X scenario's change (a linear interpolation of AEMO's answer between 0 and X MW), and each MW earns the moved price. This only goes up to F = X (100–200 MW): beyond that we would be extrapolating past what AEMO solved, and real fleets are 10–20 times larger.

In [ ]:
def fleet_revenue(frac: float) -> pd.Series:
    move = frac * ((el.down - el.base) * (el.discharged_mwh > 0) + (el.up - el.base) * (el.charged_mwh > 0))
    rev = (el.discharged_mwh - el.charged_mwh) * (el.rrp + move)
    return rev.groupby(el.region).sum() / (el.groupby("region").size() / 288) * TO_K_PER_YEAR


fracs = np.linspace(0, 1, 11)
curve = pd.DataFrame({f: fleet_revenue(f) for f in fracs})
retained = curve.div(curve[0.0], axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 4.5))
for region in REGIONS:
    xs = fracs * pairs.loc[region, "x_mw"]
    ax.plot(xs, retained.loc[region], color=REGION_COLOUR[region], marker="o", markevery=[-1], markeredgecolor=SURFACE)
    nudge = {"VIC1": 6, "QLD1": -6}.get(region, 0)   # VIC and QLD end at the same x and nearly the same y
    ax.annotate(f"{region[:-1]}  {retained.loc[region].iloc[-1]:.0f}%", (xs[-1], retained.loc[region].iloc[-1]),
                xytext=(8, nudge), textcoords="offset points", va="center", color=INK2)
ax.set_xlim(0, 240)
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.set_xlabel("fleet running TWAP's dispatch (MW)")
ax.set_ylabel("revenue per MW, % of a lone battery")
ax.set_title("A 100 MW fleet in SA would keep only about 40% of a lone battery's revenue")
plt.show()

summary = pd.DataFrame({"X (MW)": pairs.x_mw, "lone battery": curve[0.0], "fleet of X MW": curve[1.0]})
summary["drop"] = (1 - summary["fleet of X MW"] / summary["lone battery"]) * 100
summary.loc[REGIONS].round(0)

In [ ]:
fleet8 = fleet.loc[[(r, m) for r in MAINLAND for m in EL_MONTHS]].groupby(level="month").mean().mean()
compare = pd.Series({
    "TWAP, lone battery": summary.loc[MAINLAND, "lone battery"].mean(),
    "TWAP, fleet of X MW": summary.loc[MAINLAND, "fleet of X MW"].mean(),
    "real fleet, energy + FCAS": fleet8["total"],
    "real fleet, energy": fleet8["energy"],
})
fig, ax = plt.subplots(figsize=(8, 3.2))
yb = np.arange(len(compare))[::-1]
bars = ax.barh(yb, compare.values, height=0.55, color=[BLUE, BLUE, INK2, INK2], edgecolor=SURFACE)
bars[1].set_hatch("///")   # the same policy, re-priced for a fleet
for y, v in zip(yb, compare.values):
    ax.annotate(f"${v:.0f}k", (v, y), xytext=(6, 0), textcoords="offset points", va="center", color=INK2)
ax.set_yticks(yb, compare.index)
ax.grid(axis="x", color=GRID)
ax.grid(axis="y", visible=False)
ax.set_xlabel("$k per MW per year, mainland, the 8 months with sensitivity data")
ax.set_title("Price impact from a 100–200 MW fleet closes most of the gap")
plt.show()
compare.round(0).to_frame("$k/MW/yr")

**Reading it.** With only 100–200 MW running the same profile, mainland revenue per MW falls by about a third, and SA's by more than half, which is consistent with its thin market: 100 MW is a large share of SA's supply margin at times of scarcity. That alone takes TWAP from roughly 2× the fleet's energy revenue to about 15% above the fleet's energy-plus-FCAS revenue over the same months.

**Caveats.**
- **First-order only.** The sensitivities re-solve dispatch with everyone else's bids held fixed. If 100 MW of batteries really were added to a region, other participants would re-bid: other batteries would shift when they charge and discharge, and gas and other peaking units would price their offers around the new supply. The true effect could be smaller or larger than this estimate, and grows less certain the larger the fleet.
- **Small fleets only.** The linear interpolation stops at AEMO's offsets (50–200 MW), while real fleets are 1.6–3.7 GW per region, so with bids held fixed the impact at today's fleet sizes would be larger. The true price response is also a step function, not a line.
- **But real batteries don't all do the same thing at once.** A fleet's dispatch is spread across intervals and strategies, so the impact would not scale fully with fleet size either.
- **Pre-dispatch, not dispatch.** P5MIN's first interval tracks dispatch closely, but it is still a forecast run's answer.
- **Other real-world differences remain**: FCAS commitments, availability and outages, SoC policies around contracts, and batteries counted while still commissioning.

## 7. What this means for the benchmark

- **Policy comparisons are still fair.** Every policy, the TWAP and LP baselines and the perfect-foresight reference all face the same prices, so percent-of-perfect scores and rankings are like-for-like.
- **Dollar figures are a marginal-battery upper bound.** The leaderboard's \\$/MW/yr should be read as what one small price-taking battery could earn, not as fleet revenue. On the page they should carry that caveat, and for comparisons with real batteries we should quote gross revenue on unmapped prices.
- **The score leans on spikes.** Because a price-taker keeps the whole spike, much of a policy's score is decided in a few scarcity intervals (over half of TWAP's revenue in NSW and SA). Policies that hold charge for spikes are rewarded more than they would be in a market where other batteries chase the same intervals.
- **The ramp rate isn't a factor.** The benchmark's battery ramps at 100% of its capacity per minute, faster than the median grid battery: AEMO's registration data (`DUDETAIL`, August 2026) puts real batteries at a median of 40% of capacity per minute, 61% capacity-weighted. But the ramp rate only limits how far each 5-minute dispatch target can move, and at 40% per minute a battery can already swing from full charge to full discharge in one interval. Within the interval the unit moves in a straight line to its target whatever its ramp rate, so anything from 40% up gives identical dispatch.
- **Possible follow-ups.** Report a price-impact sensitivity alongside the headline score (re-price each policy's dispatch with a fleet of X MW, as here); or, for a future version, simulate with a price that responds to the battery's own dispatch using these sensitivities.